In [1]:
import pandas as pd

df = pd.read_csv("spam.csv", encoding="latin-1")   # utf-8 fails on this file
df = df[["v1", "v2"]]                               # drop the 3 junk columns
df.columns = ["label", "text"]
df["label"] = df["label"].map({"ham": 0, "spam": 1})

In [2]:
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df['text'] = df['text'].str.lower()

In [4]:
df['text']

0       go until jurong point, crazy.. available only ...
1                           ok lar... joking wif u oni...
2       free entry in 2 a wkly comp to win fa cup fina...
3       u dun say so early hor... u c already then say...
4       nah i don't think he goes to usf, he lives aro...
                              ...                        
5567    this is the 2nd time we have tried 2 contact u...
5568                will ì_ b going to esplanade fr home?
5569    pity, * was in mood for that. so...any other s...
5570    the guy did some bitching but i acted like i'd...
5571                           rofl. its true to its name
Name: text, Length: 5572, dtype: str

In [8]:
df["clean_text"] = (
    df["text"].str.lower()
    .str.replace(r"http\S+|www\S+", " ", regex=True)  # URLs first
    .str.replace(r"[^a-z\s]", " ", regex=True)        # keep only letters
    .str.replace(r"\s+", " ", regex=True)             # squeeze spaces
    .str.strip()
)

In [10]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
sw = set(stopwords.words("english"))

df["clean_text"] = df["clean_text"].apply(
    lambda t: " ".join(w for w in t.split() if w not in sw)
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hyder\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [18]:
X = df["clean_text"]
y = df["label"]

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=.2,random_state= 42
)

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [29]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
mnb = MultinomialNB(alpha=0.1)
mnb.fit(X_train_tfidf,y_train)
y_pred_train  = mnb.predict(X_train_tfidf)
y_pred_test = mnb.predict(X_test_tfidf)
print("accuracy train",accuracy_score(y_pred_train,y_train))
print("accuracy test",accuracy_score(y_pred_test,y_test))

print("the confusion matrix is :",confusion_matrix(y_test,y_pred_test))
print("the classification report is :",classification_report(y_test,y_pred_test))

accuracy train 0.9957370428539376
accuracy test 0.9829596412556054
the confusion matrix is : [[960   5]
 [ 14 136]]
the classification report is :               precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.96      0.91      0.93       150

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [30]:
from sklearn.linear_model import LogisticRegression
logistic = LogisticRegression(random_state = 42,max_iter = 2000)
logistic.fit(X_train_tfidf,y_train)

y_pred_train = logistic.predict(X_train_tfidf)
y_pred_test = logistic.predict(X_test_tfidf)
print("accuracy train",accuracy_score(y_pred_train,y_train))
print("accuracy test",accuracy_score(y_pred_test,y_test))

print("the confusion matrix is :",confusion_matrix(y_test,y_pred_test))
print("the classification report is :",classification_report(y_test,y_pred_test))

accuracy train 0.9703836661431456
accuracy test 0.9542600896860987
the confusion matrix is : [[960   5]
 [ 46 104]]
the classification report is :               precision    recall  f1-score   support

           0       0.95      0.99      0.97       965
           1       0.95      0.69      0.80       150

    accuracy                           0.95      1115
   macro avg       0.95      0.84      0.89      1115
weighted avg       0.95      0.95      0.95      1115



In [31]:
from sklearn.svm import LinearSVC
lsvc = LinearSVC()
lsvc.fit(X_train_tfidf,y_train)

y_pred_train = lsvc.predict(X_train_tfidf)
y_pred_test = lsvc.predict(X_test_tfidf)
print("accuracy train",accuracy_score(y_pred_train,y_train))
print("accuracy test",accuracy_score(y_pred_test,y_test))

print("the confusion matrix is :",confusion_matrix(y_test,y_pred_test))
print("the classification report is :",classification_report(y_test,y_pred_test))

accuracy train 0.9993269015032533
accuracy test 0.97847533632287
the confusion matrix is : [[961   4]
 [ 20 130]]
the classification report is :               precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       0.97      0.87      0.92       150

    accuracy                           0.98      1115
   macro avg       0.97      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [35]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(
    n_estimators = 500,
    learning_rate =0.1,
    max_depth=2,
    random_state = 42
)
gb.fit(X_train_tfidf,y_train)

y_pred_train = gb.predict(X_train_tfidf)
y_pred_test = gb.predict(X_test_tfidf)
print("accuracy train",accuracy_score(y_pred_train,y_train))
print("accuracy test",accuracy_score(y_pred_test,y_test))

print("the confusion matrix is :",confusion_matrix(y_test,y_pred_test))
print("the classification report is :",classification_report(y_test,y_pred_test))

accuracy train 0.9993269015032533
accuracy test 0.9730941704035875
the confusion matrix is : [[962   3]
 [ 27 123]]
the classification report is :               precision    recall  f1-score   support

           0       0.97      1.00      0.98       965
           1       0.98      0.82      0.89       150

    accuracy                           0.97      1115
   macro avg       0.97      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [36]:
from sklearn.ensemble import AdaBoostClassifier
ada = AdaBoostClassifier(
    n_estimators = 200,
    learning_rate = 1,
    random_state = 42
)
ada.fit(X_train_tfidf,y_train)

y_pred_train = ada.predict(X_train_tfidf)
y_pred_test = ada.predict(X_test_tfidf)
print("accuracy train",accuracy_score(y_pred_train,y_train))
print("accuracy test",accuracy_score(y_pred_test,y_test))

print("the confusion matrix is :",confusion_matrix(y_test,y_pred_test))
print("the classification report is :",classification_report(y_test,y_pred_test))

accuracy train 0.9470495849225937
accuracy test 0.9560538116591928
the confusion matrix is : [[959   6]
 [ 43 107]]
the classification report is :               precision    recall  f1-score   support

           0       0.96      0.99      0.98       965
           1       0.95      0.71      0.81       150

    accuracy                           0.96      1115
   macro avg       0.95      0.85      0.89      1115
weighted avg       0.96      0.96      0.95      1115



In [37]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

models = {
    "MNB (alpha=1)": MultinomialNB(),
    "MNB (alpha=0.1)": MultinomialNB(alpha=0.1),
    "Logistic Regression": LogisticRegression(),
    "LinearSVC": LinearSVC(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
}

results = []
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Spam Precision": precision_score(y_test, pred),
        "Spam Recall": recall_score(y_test, pred),
        "Spam F1": f1_score(y_test, pred),
    })

results_df = pd.DataFrame(results).sort_values("Spam F1", ascending=False)
print(results_df.round(3))

                 Model  Accuracy  Spam Precision  Spam Recall  Spam F1
1      MNB (alpha=0.1)     0.983           0.965        0.907    0.935
3            LinearSVC     0.978           0.970        0.867    0.915
4        Random Forest     0.977           1.000        0.827    0.905
0        MNB (alpha=1)     0.966           1.000        0.747    0.855
5    Gradient Boosting     0.962           0.974        0.740    0.841
2  Logistic Regression     0.954           0.954        0.693    0.803
6             AdaBoost     0.940           0.919        0.607    0.731


In [38]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(TfidfVectorizer(), MultinomialNB(alpha=0.1))
scores = cross_val_score(pipe, df["clean_text"], df["label"], cv=5, scoring="f1")
print(scores.round(3), scores.mean().round(3))

[0.955 0.916 0.921 0.93  0.913] 0.927


In [39]:
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('multinomialnb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [41]:
import re
from nltk.corpus import stopwords

sw = set(stopwords.words("english"))

def clean(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # URLs
    text = re.sub(r"[^a-z\s]", " ", text)         # keep only letters
    text = re.sub(r"\s+", " ", text).strip()      # squeeze spaces
    return " ".join(w for w in text.split() if w not in sw)

In [44]:
final = make_pipeline(TfidfVectorizer(), MultinomialNB(alpha=0.1))
final.fit(df["clean_text"], df["label"])

tests = [
    "URGENT ALERT: Your bank account has been compromised! Click this secure link immediately to verify your identity and prevent permanent suspension: http://fake-bank-login.com.",
    "CONGRATULATIONS!!! You have been selected as the grand prize winner of a luxury vacation package worth $10,000! Text 'WIN' to 555-0192 now to claim your reward!",
    "Dear Friend, I am a financial attorney with an urgent proposal regarding an unclaimed inheritance of $14.5 Million USD. Please reply with your full banking details to initiate transfer.",
    "Lose up to 20 pounds in just 5 days! No dieting, no exercise, 100% guaranteed results! Buy this miracle supplement now at an exclusive 80% discount: http://cheap-diet-pills.net.",
    "Hi Sarah, hope you're doing well. Are we still on for our sync tomorrow at 10:00 AM? Let me know if you need to reschedule.",
    "Dear Team, please find attached the monthly performance report for October. Review the figures before our board meeting on Friday.",
    "Your weekly subscription receipt for account #84920. The total amount of $14.99 has been successfully charged to your card on file.",
    "Hey! Are you guys free to grab some dinner tonight after work? There's a new Italian place down the street we should try out.",
]
# apply the same cleaning you used before (lowercase, strip URLs/punctuation, stopwords)
print(final.predict([clean(t) for t in tests]))   # 1 = spam, 0 = ham

#expected resoult [11110000]

[1 1 0 1 0 0 1 0]


In [47]:
import joblib
joblib.dump(final, "spam_model.pkl")

['spam_model.pkl']

In [49]:
final = make_pipeline(TfidfVectorizer(), MultinomialNB(alpha=0.1))
final.fit(df["clean_text"], df["label"])
joblib.dump(final, "spam_model.pkl")   # saves the trained model to a file

['spam_model.pkl']

In [50]:
import streamlit as st
import joblib, re, nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
sw = set(stopwords.words("english"))

def clean(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(w for w in text.split() if w not in sw)

@st.cache_resource
def load_model():
    return joblib.load("spam_model.pkl")

model = load_model()

st.title("📧 Spam Detector")
msg = st.text_area("Paste a message or email:")

if st.button("Check"):
    if msg.strip():
        cleaned = clean(msg)
        pred = model.predict([cleaned])[0]
        prob = model.predict_proba([cleaned])[0][1]
        if pred == 1:
            st.error(f"🚨 Spam ({prob:.0%} confidence)")
        else:
            st.success(f"✅ Not spam ({1 - prob:.0%} confidence)")
    else:
        st.warning("Type something first!")

st.caption("Trained on the SMS Spam Collection, so it works best on short, SMS-style messages.")

2026-11-06 07:42:37.170 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 07:42:37.171 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 07:42:37.172 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 07:42:37.173 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 07:42:37.175 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 07:42:37.176 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 07:42:37.177 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 07:42:37.179 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()